In [ ]:
# Removendo dependência do Google Drive conforme solicitado
import os
from pathlib import Path

In [ ]:
from pathlib import Path
import zipfile

# Pasta que será compactada
source_dir = Path("kaggle-2026-08-08-115533")

# Nome do arquivo ZIP de saída
zip_path = Path(f"{source_dir.name}.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file in source_dir.rglob("*"):
        if file.is_file():
            zipf.write(file, arcname=file.relative_to(source_dir.parent))

print(f"ZIP criado com sucesso: {zip_path.resolve()}")

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
from pathlib import Path

def extract_scalars_from_event_file(event_file):
    ea = EventAccumulator(event_file, size_guidance={'scalars': 0})
    ea.Reload()
    
    metrics = {}
    if 'scalars' in ea.Tags():
        for tag in ea.Tags()['scalars']:
            events = ea.Scalars(tag)
            metrics[tag] = pd.DataFrame(
                [(e.step, e.value) for e in events], 
                columns=['step', tag]
            )
    
    run_df = pd.DataFrame()
    for tag, df in metrics.items():
        if run_df.empty:
            run_df = df
        else:
            run_df = pd.merge(run_df, df, on='step', how='outer')
            
    return run_df

# Caminho raiz exato
base_log_dir = Path("/kaggle/working/kaggle-2026-08-08-115533/logs") 
models = ["unet", "attn_unet"]
all_data = []

print("Lendo arquivos de log do TensorBoard (3 Seeds por modelo)...")
for model in models:
    model_dir = base_log_dir / model
    if not model_dir.exists():
        continue
        
    event_files = glob.glob(os.path.join(model_dir, "**", "*tfevents*"), recursive=True)
    event_files.sort(key=os.path.getctime)
    
    for seed_idx, event_file in enumerate(event_files):
        run_df = extract_scalars_from_event_file(event_file)
        if not run_df.empty:
            run_df['model'] = model
            run_df['seed'] = seed_idx 
            all_data.append(run_df)

if all_data:
    df_master = pd.concat(all_data, ignore_index=True)
    
    # Preenchendo possíveis buracos na coluna 'epoch' usando 'ffill' (forward fill)
    if 'epoch' in df_master.columns:
        df_master['epoch'] = df_master['epoch'].ffill()
    else:
        print("Aviso: Métrica 'epoch' não encontrada. O eixo X usará os steps.")
        df_master['epoch'] = df_master['step']
    
    # As métricas exatas que você quer exibir no relatório
    target_metrics = [
        'val_loss', 
        'val_precision', 
        'val_recall', 
        'val_f1_score', 
        'val_iou'
    ]
    
    # Filtra apenas as métricas que realmente existem no DataFrame
    metrics_to_plot = [m for m in target_metrics if m in df_master.columns]
    
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.8)
    
    output_plots_dir = Path("/kaggle/working/evaluation_plots")
    output_plots_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\nGerando gráficos em 4K focados nas métricas de validação...")
    
    for metric in metrics_to_plot:
        plt.figure(figsize=(12.8, 7.2))
        
        # Mudança crucial: x='epoch' em vez de x='step'
        sns.lineplot(
            data=df_master, 
            x='epoch', 
            y=metric, 
            hue='model', 
            errorbar='sd', 
            linewidth=3.5,
            palette="viridis"
        )
        
        # Formatação bonita para o título (ex: val_f1_score -> Validação F1-Score)
        metric_clean_name = metric.replace('val_', 'Validação ').replace('_', '-').title()
        
        plt.title(f"Estabilidade Arquitetural: {metric_clean_name}\n(Média ± Desvio Padrão)", pad=20, fontweight='bold')
        plt.xlabel("Épocas", fontweight='bold')
        plt.ylabel(metric_clean_name, fontweight='bold')
        
        plt.legend(title="Arquitetura", loc="best", frameon=True, shadow=True)
        plt.tight_layout()
        
        plot_path = output_plots_dir / f"multi_seed_{metric}_4K.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.show()
else:
    print("Nenhum dado encontrado.")

In [ ]:
import datetime

# Create a unique timestamp for the session
timestamp = datetime.datetime.now().strftime("%Y-%m-%d-%H%M%S")

# Define a local output directory
drive_output_dir = Path(f"kaggle-{timestamp}")

# Create the local directory and subdirectories
drive_output_dir.mkdir(parents=True, exist_ok=True)
(drive_output_dir / "plots").mkdir(exist_ok=True)
(drive_output_dir / "models").mkdir(exist_ok=True)
(drive_output_dir / "logs").mkdir(exist_ok=True)

print(f"All outputs will be saved locally to: {drive_output_dir.absolute()}")

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path


def get_base_dir():
    """Return the repository directory on Windows, Linux, or Kaggle."""
    candidates = [
        Path.cwd() / "deforestation-unet",
        Path.cwd().parent / "deforestation-unet",
        Path("/kaggle/working/deforestation-unet"),
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find deforestation-unet. Open the notebook from its parent directory "
        "or set DEFORESTATION_UNET_DIR to the repository path."
    )


configured_base_dir = os.environ.get("DEFORESTATION_UNET_DIR")
base_dir = (
    Path(configured_base_dir).expanduser().resolve()
    if configured_base_dir
    else get_base_dir()
    )
print(f"Base directory: {base_dir}")

In [ ]:
if not (base_dir / "requirements.txt").is_file():
    raise FileNotFoundError(f"requirements.txt was not found in {base_dir}")

print(f"Using local repository: {base_dir}")

In [ ]:
%pip install -r {base_dir}/requirements.txt
%pip install albumentations

In [ ]:
import albumentations as A
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio as rio
import rasterio.plot as rplt

In [ ]:
import os

import numpy as np
import torch


def configure_device():
    if not torch.cuda.is_available():
        return torch.device("cpu"), 0

    gpu_count = torch.cuda.device_count()
    torch.backends.cudnn.benchmark = True
    if hasattr(torch.backends.cuda.matmul, "allow_tf32"):
        torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
    return torch.device("cuda:0"), gpu_count


device, gpu_count = configure_device()
ddp_world_size = gpu_count if gpu_count > 1 else 1
use_ddp = ddp_world_size > 1
use_data_parallel = False
print(f"Using device: {device}; CUDA GPUs available: {gpu_count}.")
if use_ddp:
    print(f"DDP training is configured for {ddp_world_size} GPU processes.")


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_seed(42)
torch.backends.cudnn.deterministic = False

# Each DDP process gets its own DataLoader; divide CPU readers across ranks.
cpu_count = os.cpu_count() or 1
total_worker_budget = min(8, max(4, cpu_count - 1))
default_workers = max(1, total_worker_budget // ddp_world_size)
num_workers = int(os.environ.get("NUM_WORKERS", default_workers))
batch_size_per_gpu = int(os.environ.get("BATCH_SIZE_PER_GPU", 4))
batch_size = batch_size_per_gpu * ddp_world_size
pin_memory = device.type == "cuda"
prefetch_factor = 2
print(
    f"DDP workers per process: {num_workers}; per-GPU batch size: {batch_size_per_gpu}; "
    f"global batch size: {batch_size}."
  )

In [ ]:
CHECKPOINT_PATH = drive_output_dir / "models" / "checkpoints"
CHECKPOINT_PATH.mkdir(parents=True, exist_ok=True)

# Define log directory for consistency
LOG_PATH = drive_output_dir / "logs"
LOG_PATH.mkdir(parents=True, exist_ok=True)

paths = {
    "AM4": base_dir / "data" / "AM4",
}

downloads = {
    "AM4": ["https://doi.org/10.5281/zenodo.4498086", "AMAZON.rar"]
}

for path in paths.values():
    path.mkdir(parents=True, exist_ok=True)

print(f"Checkpoints will be saved to: {CHECKPOINT_PATH.absolute()}")

In [ ]:
# Download and extract datasets.
# patool is only needed when an archive must be extracted.
import patoolib

for dataset, (url, filename) in downloads.items():
    dataset_path = paths[dataset]
    local_archive_path = dataset_path / filename

    if local_archive_path.exists():
        archive_path = local_archive_path
        print(f"{dataset}: found archive at {archive_path}.")
    else:
        print(f"Downloading {dataset} dataset...")
        subprocess.run(
            ["zenodo_get", "-o", str(dataset_path), "-g", filename, url],
            check=True,
        )
        archive_path = local_archive_path
        print(f"{dataset} downloaded to {dataset_path}")

    extracted_dir_name = Path(filename).stem
    final_dataset_dir = dataset_path / extracted_dir_name.lower().replace(" ", "_")
    if final_dataset_dir.exists():
        print(f"{dataset} already extracted. Skipping extraction.")
        continue

    print(f"Extracting {dataset}...")
    patoolib.extract_archive(str(archive_path), outdir=str(dataset_path))

    extracted_path = dataset_path / extracted_dir_name
    if extracted_path.exists() and extracted_path != final_dataset_dir:
        extracted_path.rename(final_dataset_dir)

    print(f"{dataset} extracted to {final_dataset_dir}")

# Data validation and exploration

In [ ]:
train_dataset_dir = paths["AM4"] / "amazon" / "Training"
test_dataset_dir = paths["AM4"] / "amazon" / "Test"
val_dataset_dir = paths["AM4"] / "amazon" / "Validation"

train_dataset_dir_imgs = train_dataset_dir / "image"
test_dataset_dir_imgs = test_dataset_dir / "image"
val_dataset_dir_imgs = val_dataset_dir / "images"

train_dataset_dir_masks = train_dataset_dir / "label"
test_dataset_dir_masks = test_dataset_dir / "mask"
val_dataset_dir_masks = val_dataset_dir / "masks"

img_dirs = [train_dataset_dir_imgs, test_dataset_dir_imgs, val_dataset_dir_imgs]
mask_dirs = [train_dataset_dir_masks, test_dataset_dir_masks, val_dataset_dir_masks]

In [ ]:
img = "S2A_MSIL2A_20200111T142701_N0213_R053_T20NQG_20200111T164651_01_07.tif"
img_path = train_dataset_dir_imgs / img
mask_path = train_dataset_dir_masks / img
src = rio.open(img_path)
print("Image metadata:")
for key, value in src.meta.items():
    print(f"{key}: {value}")
print("\nCRS information:")
for key,value in src.meta["crs"].items():
    print(f"{key}: {value}")
print(f"Spatial resolution is {src.res} meters.")
print(f"Bounds of the area are {src.bounds}, expressed in meters.")


img2 = 'S2A_MSIL2A_20200111T142701_N0213_R053_T20NQG_20200111T164651_04_07.tif'
img_path2 = train_dataset_dir_imgs / img2
mask_path2 = train_dataset_dir_masks / img2
src2 = rio.open(img_path2)



In [ ]:
# NIR band
src.read(4)
print("Image profile:")
for key, value in src.profile.items():
    print(f"{key}: {value}")

In [ ]:
def save_plot(fig, filename):
    # Ensure plots are saved inside the timestamped drive directory
    filepath = drive_output_dir / "plots" / filename
    filepath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(filepath)

    # Display the plot in the notebook before closing
    plt.show()

    plt.close(fig)
    print(f"Plot saved to: {filepath}")

def plot_nrg(src):
    # Read bands
    n = src.read(4)
    r = src.read(1)
    g = src.read(2)

    # Stack into NRG image
    nrg = np.dstack((n, r, g))

    # Normalize to 0-1
    nrg = nrg.astype(float) / 6000

    # Clip values
    nrg = np.clip(nrg, 0, 1)

    fig = plt.figure(figsize=(7,7))
    plt.imshow(nrg)
    plt.title("NRG Image")
    plt.xlabel("Column #")
    plt.ylabel("Row #")
    save_plot(fig, "NRG_Image.png")

In [ ]:
def make_img_plots(src, plot_id):
    red_band = src.read(1).astype('float32')
    green_band = src.read(2).astype('float32')
    nir_band = src.read(4).astype('float32')

    # NDVI
    denominator = nir_band + red_band
    ndvi = np.where(denominator == 0, 0, (nir_band - red_band) / denominator)
    ndvi = np.clip(ndvi, -1, 1)


    # Stack into NRG image, normlize and clip
    nrg = np.dstack((nir_band, red_band, green_band))
    nrg = nrg.astype(float) / 6000
    nrg = np.clip(nrg, 0, 1)

    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    axes = axes.flatten()

    # Plot RGB composite
    rplt.show(src, ax=axes[0])
    axes[0].set_title("RGB Composite")

    #Plot NIR band
    im = axes[1].imshow(nir_band, cmap='Reds')
    #rplt.show((src, 4), ax=axes[1], cmap='Reds')
    axes[1].set_title("NIR Band")
    plt.colorbar(im, ax=axes[1], shrink=0.7)

    # Plot NDVI
    #rplot.show(ndvi, ax=axes[2], cmap='RdYlGn', vmin=-1, vmax=1)
    im = axes[2].imshow(ndvi, cmap="RdYlGn", vmin=-1, vmax=1)
    axes[2].set_title("NDVI")
    plt.colorbar(im, ax=axes[2], shrink=0.7)

    #plot NRG
    im = axes[3].imshow(nrg, cmap="RdYlGn", vmin=-1, vmax=1)
    axes[3].set_title("NRG Image")

    plt.tight_layout()
    save_plot(fig, f"image_plots_{plot_id}.png")

make_img_plots(src, "src")

Photosinthesizing plants reflect NIR stronger than barren soil, so NIR is more intense in forest. Clouds are dark here, as cloud reflect NIR light. Shaadows of clouds are darker, as less light is reaching the surface and therefore there is less reflectance of NIR light.

NDVI is a measure of plant health, also more intense in forested regions. Clouds have 0 NDVI as they reflect both NIR aand red light strongly. Shadows  of clouds equally have NDVI close to 0.

Finally, NRG plot shows NIR in red, red in green and green in blue. They are useful for tracking seasonal changes. Photosinthesizing plants reflect a lot of NIR and some green, not a lot of red. Therefore, forested regions are bright red. Less forested regions are colored less bright, since their reflection of NIR is more balanced with the reflection of other colors (compared with forested regions).
Clouds, which reflect all colors, are white. Shadows are dark as there is  less refelctance due to less light hitting the surface.

In [ ]:
make_img_plots(src2, "src2")

In [ ]:
#inspecting image and mask alignment, by plotting the image and mask for the same file name overlayed

fig, axes = plt.subplots(1,2, figsize=(10, 10))

mask = mask_path2
src_mask = rio.open(mask)
rplt.show(src2, ax=axes[0])
rplt.show(src2, ax=axes[1])
axes[0].set_title("Image")
rplt.show(src_mask, ax=axes[1], alpha=0.2, cmap="Greens")
axes[1].set_title("Mask overlayed on image")

#checking that the bounding boxes of the image and mask are the same
print(f"Image bounds: {src2.bounds}")
print(f"Mask bounds: {src_mask.bounds}")
save_plot(fig, "image_mask_alignment.png")

In [ ]:
from shapely import box


def centroid(bounds):
    return (
        (bounds.left + bounds.right) / 2,
        (bounds.top + bounds.bottom) / 2,
    )


train_image_dict_am4 = {img.stem: img.name for img in train_dataset_dir_imgs.glob("*.tif")}
train_mask_dict_am4 = {label.stem: label.name for label in train_dataset_dir_masks.glob("*.tif")}
test_image_dict_am4 = {img.stem: img.name for img in test_dataset_dir_imgs.glob("*.tif")}
test_mask_dict_am4 = {label.stem: label.name for label in test_dataset_dir_masks.glob("*.tif")}
val_image_dict_am4 = {img.stem: img.name for img in val_dataset_dir_imgs.glob("*.tif")}
val_mask_dict_am4 = {label.stem: label.name for label in val_dataset_dir_masks.glob("*.tif")}
brightness_scores = {}
images_to_remove = set()


def validate_metadata(image_path, mask_path, label):
    try:
        with rio.open(image_path) as image, rio.open(mask_path) as mask:
            valid = (
                image.bounds == mask.bounds
                and image.crs == mask.crs
                and image.res == mask.res
                and image.width == mask.width
                and image.height == mask.height
                and image.count == 4
                and mask.count == 1
                and image.dtypes[0] == "uint16"
                and mask.dtypes[0] == "uint8"
            )
            if not valid:
                print(f"Metadata validation failed for {label}.")
                images_to_remove.add(image_path)
                return False
    except rio.errors.RasterioError as error:
        print(f"Could not open the image/mask pair for {label}: {error}")
        images_to_remove.add(image_path)
        return False
    return True


def validate_image_mask(image_path, mask_path, label):
    with rio.open(image_path) as image, rio.open(mask_path) as mask:
        pixels = image.read([1, 2, 3])
        brightness_scores[str(image_path)] = np.sum(pixels.sum(axis=0) > 5000)
        if not np.any(pixels > 0) or not np.any(mask.read(1) > 0):
            print(f"Image or mask for {label} contains only zeros.")
            images_to_remove.add(image_path)
            return None
        return centroid(image.bounds)


def validate_split(image_directory, mask_directory, image_dict, mask_dict, split_name):
    labels, centroids = [], []
    print(f"Validating {split_name} images and masks...")
    for label, image_name in image_dict.items():
        if label not in mask_dict:
            print(f"Image {image_name} has no matching mask.")
            images_to_remove.add(image_directory / image_name)
            continue
        image_path = image_directory / image_name
        mask_path = mask_directory / mask_dict[label]
        if validate_metadata(image_path, mask_path, label):
            center = validate_image_mask(image_path, mask_path, label)
            if center is not None:
                labels.append(label)
                centroids.append(center)
    return labels, centroids


train_labels, train_centroids = validate_split(
    train_dataset_dir_imgs, train_dataset_dir_masks, train_image_dict_am4, train_mask_dict_am4, "training"
    )
test_labels, test_centroids = validate_split(
    test_dataset_dir_imgs, test_dataset_dir_masks, test_image_dict_am4, test_mask_dict_am4, "test"
    )
val_labels, val_centroids = validate_split(
    val_dataset_dir_imgs, val_dataset_dir_masks, val_image_dict_am4, val_mask_dict_am4, "validation"
    )

In [ ]:
def make_centroid_gdf(labels, centroids):
    geometry = gpd.points_from_xy(
        [centroid[0] for centroid in centroids],
        [centroid[1] for centroid in centroids],
    )
    return gpd.GeoDataFrame({"label": labels}, geometry=geometry)


train_gdf = make_centroid_gdf(train_labels, train_centroids)
test_gdf = make_centroid_gdf(test_labels, test_centroids)
val_gdf = make_centroid_gdf(val_labels, val_centroids)

for dataframe in (train_gdf, test_gdf, val_gdf):
    dataframe["buffer"] = dataframe.geometry.buffer(5)
    dataframe.set_geometry("buffer", inplace=True)

In [ ]:
sindex = train_gdf.sindex

possible_matches = val_gdf.geometry.apply(
    lambda geom: list(sindex.intersection(geom.bounds))
)
for idx, geom in enumerate(val_gdf.geometry):
    candidates = sindex.intersection(geom.bounds)
    for i in candidates:
        if geom.intersects(train_gdf.geometry[i]):
            print(f"Validation image {val_gdf.label[idx]} overlaps with training image {train_gdf.label[i]}.")


In [ ]:
#plotting top 5 images with highest brightness scores
top_5_brightest = sorted(brightness_scores.items(), key=lambda x: x[1], reverse=True)[:5]
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for i in range(5):
    label = top_5_brightest[i][0]
    print(f"Label: {label}, Brightness Score: {brightness_scores[label]}")
    src = rio.open(label)
    rplt.show(src, ax=axes[i])
save_plot(fig, "top_5_brightest_images.png")

In [ ]:
image_paths = sorted(train_dataset_dir_imgs.glob("*.tif"))
print(f"Found {len(image_paths)} training images.")

bins = np.arange(0, 6001, 25)
bin_centers = (bins[:-1] + bins[1:]) / 2
band_indexes = {"Red": 1, "Green": 2, "Blue": 3, "NIR": 4}

for color, band_index in band_indexes.items():
    hist_counts = np.zeros(len(bins) - 1, dtype=np.int64)
    for image_path in image_paths:
        with rio.open(image_path) as source:
            band = source.read(band_index).astype(np.int32)
        counts, _ = np.histogram(band, bins=bins)
        hist_counts += counts

    fig = plt.figure(figsize=(10, 3))
    plt.bar(bin_centers, hist_counts, width=np.diff(bins), align="center", edgecolor="black")
    plt.xlabel(f"{color} band pixel value")
    plt.ylabel("Pixel count")
    plt.title(f"Histogram of {color.lower()} band values for AM4 training images")
    plt.xlim(bins[0], bins[-1])
    plt.tight_layout()
    save_plot(fig, f"histogram_{color}_band.png")

In [ ]:
image_paths = sorted(train_dataset_dir_imgs.glob("*.tif"))
print(f"Found {len(image_paths)} training images.")

red_values = []
for img_path in image_paths:
    with rio.open(img_path) as src:
        red_values.append(src.read(1).astype(np.int32).ravel())

red_values = np.concatenate(red_values)

fig = plt.figure(figsize=(10, 4))
plt.boxplot(red_values, vert=False, patch_artist=True,
            boxprops=dict(facecolor="lightblue", color="black"),
            medianprops=dict(color="red"))
plt.xlabel("Red band pixel value")
plt.title("Boxplot of red band values across AM4 training images")
plt.grid(axis="x", linestyle="--", alpha=0.5)
save_plot(fig, "boxplot_red_band.png")

In [ ]:
green_values = []
for img_path in image_paths:
    with rio.open(img_path) as src:
        green_values.append(src.read(2).astype(np.int32).ravel())

green_values = np.concatenate(green_values)

fig = plt.figure(figsize=(10, 4))
plt.boxplot(green_values, vert=False, patch_artist=True,
            boxprops=dict(facecolor="lightgreen", color="black"),
            medianprops=dict(color="darkgreen"))
plt.xlabel("Green band pixel value")
plt.title("Boxplot of green band values across AM4 training images")
plt.grid(axis="x", linestyle="--", alpha=0.5)
save_plot(fig, "boxplot_green_band.png")

In [ ]:
blue_values = []
for img_path in image_paths:
    with rio.open(img_path) as src:
        blue_values.append(src.read(3).astype(np.int32).ravel())

blue_values = np.concatenate(blue_values)

fig = plt.figure(figsize=(10, 4))
plt.boxplot(blue_values, vert=False, patch_artist=True,
            boxprops=dict(facecolor="lightblue", color="black"),
            medianprops=dict(color="darkblue"))
plt.xlabel("Blue band pixel value")
plt.title("Boxplot of blue band values across AM4 training images")
plt.grid(axis="x", linestyle="--", alpha=0.5)
save_plot(fig, "boxplot_blue_band.png")

In [ ]:
nir_values = []
for img_path in image_paths:
    with rio.open(img_path) as src:
        nir_values.append(src.read(4).astype(np.int32).ravel())

nir_values = np.concatenate(nir_values)

fig = plt.figure(figsize=(10, 4))
plt.boxplot(nir_values, vert=False, patch_artist=True,
            boxprops=dict(facecolor="lightblue", color="black"),
            medianprops=dict(color="darkblue"))
plt.xlabel("NIR band pixel value")
plt.title("Boxplot of NIR band values across AM4 training images")
plt.grid(axis="x", linestyle="--", alpha=0.5)
save_plot(fig, "boxplot_nir_band.png")

In [ ]:
#find which image(s) have rgb values above 10000
for img_path in image_paths:
    with rio.open(img_path) as src:
        red_band = src.read(1).astype(np.int32)
        green_band = src.read(2).astype(np.int32)
        blue_band = src.read(3).astype(np.int32)
        nir_band = src.read(4).astype(np.int32)
        if np.any(red_band > 10000):
            print(f"Image {img_path.name} has red values above 10000 at location {np.where(red_band > 10000)}.")
            images_to_remove.add(Path(img_path))
        if np.any(green_band > 10000):
            print(f"Image {img_path.name} has green values above 10000 at location {np.where(green_band > 10000)}.")
            images_to_remove.add(Path(img_path))
        if np.any(blue_band > 10000):
            print(f"Image {img_path.name} has blue values above 10000 at location {np.where(blue_band > 10000)}.")
            images_to_remove.add(Path(img_path))
        if np.any(nir_band > 10000):
            print(f"Image {img_path.name} has NIR values above 10000 at location {np.where(nir_band > 10000)}.")
            images_to_remove.add(Path(img_path))

In [ ]:
# Move invalid image-mask pairs together so later dataset loading remains consistent.
for image_path in sorted(images_to_remove):
    image_path = Path(image_path)
    error_folder = image_path.parent / "error"
    error_folder.mkdir(exist_ok=True)

    paired_mask_directories = {
        train_dataset_dir_imgs: train_dataset_dir_masks,
        test_dataset_dir_imgs: test_dataset_dir_masks,
        val_dataset_dir_imgs: val_dataset_dir_masks,
    }
    mask_directory = paired_mask_directories.get(image_path.parent)
    candidates = [image_path]
    if mask_directory is not None:
        candidates.append(mask_directory / image_path.name)

    for source_path in candidates:
        if source_path.exists():
            destination = source_path.parent / "error" / source_path.name
            destination.parent.mkdir(exist_ok=True)
            shutil.move(str(source_path), str(destination))
            print(f"Moved invalid file to {destination}")

## Dataset quality concerns

RGB:
- 60 imgs from sentinel-2 satelite imagery
- max 30% cloud coverage in each picture
- 8-BIT geoTiff format?
- sr: 10m/pixel
- forest masks obtained from a land use classification process using GRASS-GIS 7.6.1 software -> automated
AM4:


# Creating the dataset class and data loader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class DeforestationDataset(Dataset):

    def __init__(self, root_directory,image_name,label_name,indices = None, transform=None):
        self.root_directory = root_directory
        self.image_directory = self.root_directory / str(image_name)
        self.label_directory = self.root_directory / str(label_name)
        self.image_paths = sorted(self.image_directory.glob("*.tif"))
        self.label_paths = sorted(self.label_directory.glob("*.tif"))
        self.image_dict = {img.name.split('.')[0]: img.name for img in self.image_paths}
        self.label_dict = {lbl.name.split('.')[0]: lbl.name for lbl in self.label_paths}

        for key,value in self.image_dict.items():
            if key not in self.label_dict.keys():
                print(f"Warning: image {value} does not have a corresponding label.")

        self.transform = transform
        self.indices = indices

    def __len__(self):
        if self.indices:
            dataset_size = len(self.indices)
        else:
            dataset_size = len(self.image_paths)
        return dataset_size

    def __getitem__(self, index):
        if self.indices:
            img_path = self.image_paths[self.indices[index]]
            img_name = img_path.name
            lbl_path = self.label_directory / img_name
        else:
            img_path = self.image_paths[index]
            img_name = img_path.name
            lbl_path = self.label_directory / img_name

        with rio.open(img_path) as img:
            img = img.read().astype('float32')
            if self.transform:
                img = self.transform(img)

            lbl = rio.open(lbl_path)
            lbl = lbl.read().astype('float32')

            sample = [img,torch.from_numpy(lbl)]
            return sample

# Adjust batch_size for cloud environments (like Kaggle or Colab)
# The user reported OutOfMemoryError, so keeping a smaller batch_size for cloud environments is prudent.
batch_size = 4 # Default for cloud environments
num_workers = 2

In [ ]:
from torch.utils.data import DataLoader, Dataset


class DeforestationDataset(Dataset):
    def __init__(
        self, root_directory, image_name, label_name, indices=None, transform=None, augmentation=None
    ):
        self.root_directory = Path(root_directory)
        self.image_directory = self.root_directory / image_name
        self.label_directory = self.root_directory / label_name
        self.image_paths = sorted(self.image_directory.glob("*.tif"))
        self.indices = indices
        self.transform = transform
        self.augmentation = augmentation

        missing_masks = [
            path.name for path in self.image_paths if not (self.label_directory / path.name).is_file()
        ]
        if missing_masks:
            raise FileNotFoundError(
                f"{len(missing_masks)} image(s) have no matching mask in {self.label_directory}; "
                f"for example: {missing_masks[0]}"
            )

    def __len__(self):
        return len(self.indices) if self.indices is not None else len(self.image_paths)

    def __getitem__(self, index):
        image_index = self.indices[index] if self.indices is not None else index
        image_path = self.image_paths[image_index]
        label_path = self.label_directory / image_path.name

        with rio.open(image_path) as image_file:
            image = image_file.read().astype("float32")
        with rio.open(label_path) as label_file:
            label = label_file.read(1).astype("float32")

        if self.augmentation is not None:
            augmented = self.augmentation(
                image=np.moveaxis(image, 0, -1),
                mask=label,
            )
            image = np.moveaxis(augmented["image"], -1, 0)
            label = augmented["mask"]

        if self.transform is not None:
            image = self.transform(image)

        if label.max() > 1:
            label /= label.max()
        return image, torch.from_numpy(np.ascontiguousarray(label)).unsqueeze(0)


class MinMaxScale:
    def __call__(self, image):
        image_min, image_max = np.min(image), np.max(image)
        return (image - image_min) / (image_max - image_min + 1e-8)


train_augmentation = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=10, p=0.5),
        # Simula variações de irradiância solar e cobertura de nuvens finas.
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        # Simula alterações de gama causadas por aerossóis e fumaça de queimadas.
        A.RandomGamma(gamma_limit=(80, 120), p=0.4),
        # Simula ruído de sensor em condições atmosféricas adversas.
        A.GaussNoise(std_range=(0.01, 0.05), p=0.3),
    ]
)
train_transform = MinMaxScale()
train_dataset_am4 = DeforestationDataset(
    base_dir / "data" / "AM4" / "amazon" / "Training",
    "image",
    "label",
    transform=train_transform,
    augmentation=train_augmentation,
)
test_dataset_am4 = DeforestationDataset(
    base_dir / "data" / "AM4" / "amazon" / "Test", "image", "mask", transform=train_transform
)
val_dataset_am4 = DeforestationDataset(
    base_dir / "data" / "AM4" / "amazon" / "Validation", "images", "masks", transform=train_transform
)

if not len(train_dataset_am4):
    raise RuntimeError("No training images were found. Run the dataset download/extract cell first.")

loader_options = {
    "batch_size": batch_size,
    "num_workers": num_workers,
    "pin_memory": pin_memory,
    "persistent_workers": num_workers > 0,
}
if num_workers > 0:
    loader_options["prefetch_factor"] = prefetch_factor

train_dataloader_am4 = DataLoader(
    train_dataset_am4, shuffle=True, **loader_options
)
test_dataloader_am4 = DataLoader(test_dataset_am4, shuffle=False, **loader_options)
val_dataloader_am4 = DataLoader(val_dataset_am4, shuffle=False, **loader_options)

sample_image, sample_mask = train_dataset_am4[0]
print(
    f"Dataset sizes - train: {len(train_dataset_am4)}, validation: {len(val_dataset_am4)}, "
    f"test: {len(test_dataset_am4)}"
)
print(f"Sample shapes - image: {tuple(sample_image.shape)}, mask: {tuple(sample_mask.shape)}")

# U-net and Attention U-net architectures

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, n_input, n_output, dropout_probability=0.5):
        super().__init__()
        self.functions = nn.Sequential(
            nn.Conv2d(n_input, n_output, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(n_output),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout_probability),
            nn.Conv2d(n_output, n_output, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(n_output),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout_probability),
        )

    def forward(self, inputs):
        return self.functions(inputs)


def enable_mc_dropout(model):
    """Keep dropout stochastic while BatchNorm retains evaluation statistics."""
    for module in model.modules():
        if isinstance(module, nn.Dropout2d):
            module.train()


class Downsample(nn.Module):
    def __init__(self, n_input, n_output):
        super().__init__()
        self.double_conv = DoubleConv(n_input, n_output)
        self.pool = nn.MaxPool2d(kernel_size=2)

    def forward(self, inputs):
        hidden = self.double_conv(inputs)
        return hidden, self.pool(hidden)


class Upsample(nn.Module):
    def __init__(self, n_input, n_output):
        super().__init__()
        self.conv_transpose = nn.ConvTranspose2d(n_input, n_output, kernel_size=2, stride=2)
        self.double_conv = DoubleConv(n_input, n_output)

    def forward(self, hidden, inputs):
        inputs = self.conv_transpose(inputs)
        inputs = torch.cat([inputs, hidden], dim=1)
        return self.double_conv(inputs)


class UNet(nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        self.downsample1 = Downsample(num_inputs, 64)
        self.downsample2 = Downsample(64, 128)
        self.downsample3 = Downsample(128, 256)
        self.downsample4 = Downsample(256, 512)
        self.bottleneck = DoubleConv(512, 1024)
        self.upsample1 = Upsample(1024, 512)
        self.upsample2 = Upsample(512, 256)
        self.upsample3 = Upsample(256, 128)
        self.upsample4 = Upsample(128, 64)
        self.conv2nd = nn.Conv2d(64, num_outputs, kernel_size=1)

    def forward(self, inputs):
        hidden1, inputs = self.downsample1(inputs)
        hidden2, inputs = self.downsample2(inputs)
        hidden3, inputs = self.downsample3(inputs)
        hidden4, inputs = self.downsample4(inputs)
        inputs = self.bottleneck(inputs)
        inputs = self.upsample1(hidden4, inputs)
        inputs = self.upsample2(hidden3, inputs)
        inputs = self.upsample3(hidden2, inputs)
        inputs = self.upsample4(hidden1, inputs)
        return self.conv2nd(inputs)

In [ ]:
class AttentionGate(nn.Module):
    def __init__(self,n_input_x,n_input_g):
        super().__init__()
        self.conv_x = nn.Conv2d(n_input_x, n_input_x, kernel_size=1, stride = 2, padding = 0) #figure out stride and padding so that its resolutions changes to that of g
        self.conv_g = nn.Conv2d(n_input_g, n_input_x, kernel_size=1, stride = 1, padding = 0)
        self.relu = nn.ReLU(inplace=True)
        self.conv_attn = nn.Conv2d(n_input_x, 1, kernel_size=1, stride=1, padding=0)
        self.sigmoid = nn.Sigmoid()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)


    def forward(self,x,g):
        y = self.conv_x(x)
        g = self.conv_g(g)
        y = self.relu(y+g)
        attn = self.conv_attn(y)
        attn = self.sigmoid(attn)
        attn = self.upsample(attn)
        y = attn * x
        return y

class AttentionUNet(nn.Module):
    def __init__(self,num_inputs, num_outputs):
        super().__init__()
        self.downsample1 = Downsample(num_inputs,16)
        self.downsample2 = Downsample(16,32)
        self.downsample3 = Downsample(32,64)
        self.downsample4 = Downsample(64,128)
        self.bottleneck = DoubleConv(128,256)
        self.upsample1 = Upsample(256,128)
        self.upsample2 = Upsample(128,64)
        self.upsample3 = Upsample(64,32)
        self.upsample4 = Upsample(32,16)
        self.conv2nd = nn.Conv2d(16,num_outputs,kernel_size=1)
        self.ag1 = AttentionGate(128,256)
        self.ag2 = AttentionGate(64,128)
        self.ag3 = AttentionGate(32,64)
        self.ag4 = AttentionGate(16,32)

    def forward(self,x):
        g1,x = self.downsample1(x)
        g2,x = self.downsample2(x)
        g3,x = self.downsample3(x)
        g4,x = self.downsample4(x)
        x = self.bottleneck(x)
        g4 = self.ag1(g4,x)
        x = self.upsample1(g4,x)
        g3 = self.ag2(g3,x)
        x = self.upsample2(g3,x)
        g2 = self.ag3(g2,x)
        x = self.upsample3(g2,x)
        g1 = self.ag4(g1,x)
        x = self.upsample4(g1,x)
        x = self.conv2nd(x)
        return x


# attUnet = AttentionUNet(num_inputs = 4,num_outputs = 1)
# sample = next(iter(train_dataloader_am4))[0]
# output = attUnet(sample)

In [ ]:
# Run this once after defining the architectures to verify tensor shapes and MC Dropout.
with torch.no_grad():
    smoke_input = torch.zeros(1, 4, 32, 32, device=device)
    for model_class in (UNet, AttentionUNet):
        smoke_model = model_class(num_inputs=4, num_outputs=1).to(device).eval()
        enable_mc_dropout(smoke_model)
        assert any(
            module.training for module in smoke_model.modules() if isinstance(module, nn.Dropout2d)
        )
        smoke_output = smoke_model(smoke_input)
        assert smoke_output.shape == (1, 1, 32, 32), smoke_output.shape
        del smoke_model

del smoke_input
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Architecture and MC Dropout smoke test passed.")

In [ ]:
class FocalDiceLoss(nn.Module):
    """Focal Loss + Soft Dice Loss.

    O termo Focal (α, γ) penaliza mais os pixels de borda difíceis e os
    falsos negativos (regiões de desmatamento subdetectadas), enquanto o
    Dice garante sobreposição global máscara/predição.

    Args:
        alpha: Fator de balanceamento para a classe positiva (desmatamento).
               Valores > 0.5 aumentam o peso dos pixels de floresta perdida.
        gamma: Expoente de foco. γ = 0 reduz ao BCE padrão; γ = 2 é o
               valor canônico da literatura (Lin et al., 2017).
        smooth: Constante de suavização do Dice para estabilidade numérica.
        dice_weight: Peso relativo do Dice em relação ao Focal Loss.
    """

    def __init__(
        self,
        alpha: float = 0.75,
        gamma: float = 2.0,
        smooth: float = 1.0,
        dice_weight: float = 1.0,
    ):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smooth = smooth
        self.dice_weight = dice_weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # ── Focal Loss ────────────────────────────────────────────────────────
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probabilities = torch.sigmoid(logits)
        p_t = probabilities * targets + (1 - probabilities) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_loss = (alpha_t * (1 - p_t) ** self.gamma * bce).mean()

        # ── Soft Dice Loss ────────────────────────────────────────────────────
        dimensions = tuple(range(1, logits.ndim))
        intersection = (probabilities * targets).sum(dim=dimensions)
        denominator = probabilities.sum(dim=dimensions) + targets.sum(dim=dimensions)
        dice_loss = 1 - ((2 * intersection + self.smooth) / (denominator + self.smooth))

        return focal_loss + self.dice_weight * dice_loss.mean()


def unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model


from typing import Any


def train_and_eval(
    model_name: str,
    model_hparams: dict[str, Any],
    optimizer_name: str,
    optimizer_hparams: dict[str, Any],
    num_epochs: int,
    train_dataloader,
    val_dataloader,
    attention: bool,
    seed: int = 0,
    use_multi_gpu: bool = use_data_parallel,
    device_override: torch.device = device,
    early_stopping_patience: int = 30,
    early_stopping_min_delta: float = 0.0,
):
    saved_model = CHECKPOINT_PATH / f"{model_name}_{seed}.pt"
    set_seed(seed)

    model_class = AttentionUNet if attention else UNet
    model = model_class(**model_hparams).to(device_override)
    if use_multi_gpu and device_override.type == "cuda":
        model = nn.DataParallel(model)

    optimizer = getattr(torch.optim, optimizer_name)(model.parameters(), **optimizer_hparams)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
    )
    criterion = FocalDiceLoss(alpha=0.75, gamma=2.0)

    amp_enabled = device_override.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

    best_val_loss = float("inf")
    epochs_without_improvement = 0
    train_metrics = [[] for _ in range(5)]
    val_metrics = [[] for _ in range(5)]

    for epoch in range(num_epochs):
        model.train()
        train_loss = total_tp = total_fp = total_fn = 0.0

        for images, masks in train_dataloader:
            images = images.to(device_override, non_blocking=pin_memory)
            masks = masks.to(device_override, non_blocking=pin_memory)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=device_override.type, enabled=amp_enabled):
                outputs = model(images)
                if outputs.shape != masks.shape:
                    raise RuntimeError(
                        f"Output/mask shape mismatch: {outputs.shape} vs {masks.shape}"
                    )
                loss = criterion(outputs, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * images.size(0)
            predictions = (outputs.detach() > 0).float()
            total_tp += (predictions * masks).sum().item()
            total_fp += (predictions * (1 - masks)).sum().item()
            total_fn += ((1 - predictions) * masks).sum().item()

        train_loss /= len(train_dataloader.dataset)
        train_precision = total_tp / (total_tp + total_fp + 1e-8)
        train_recall = total_tp / (total_tp + total_fn + 1e-8)
        train_f1 = 2 * train_precision * train_recall / (train_precision + train_recall + 1e-8)
        train_iou = total_tp / (total_tp + total_fp + total_fn + 1e-8)
        train_metrics[0].append(train_loss)
        train_metrics[1].append(train_precision)
        train_metrics[2].append(train_recall)
        train_metrics[3].append(train_f1)
        train_metrics[4].append(train_iou)

        model.eval()
        val_loss = total_tp = total_fp = total_fn = 0.0

        with torch.no_grad():
            for images, masks in val_dataloader:
                images = images.to(device_override, non_blocking=pin_memory)
                masks = masks.to(device_override, non_blocking=pin_memory)

                with torch.amp.autocast(device_type=device_override.type, enabled=amp_enabled):
                    outputs = model(images)
                    loss = criterion(outputs, masks)

                val_loss += loss.item() * images.size(0)
                predictions = (outputs > 0).float()
                total_tp += (predictions * masks).sum().item()
                total_fp += (predictions * (1 - masks)).sum().item()
                total_fn += ((1 - predictions) * masks).sum().item()

        val_loss /= len(val_dataloader.dataset)
        scheduler.step(val_loss)

        val_precision = total_tp / (total_tp + total_fp + 1e-8)
        val_recall = total_tp / (total_tp + total_fn + 1e-8)
        val_f1 = 2 * val_precision * val_recall / (val_precision + val_recall + 1e-8)
        val_iou = total_tp / (total_tp + total_fp + total_fn + 1e-8)
        val_metrics[0].append(val_loss)
        val_metrics[1].append(val_precision)
        val_metrics[2].append(val_recall)
        val_metrics[3].append(val_f1)
        val_metrics[4].append(val_iou)

        learning_rate = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"train loss {train_loss:.4f}, F1 {train_f1:.4f}, IoU {train_iou:.4f} | "
            f"val loss {val_loss:.4f}, F1 {val_f1:.4f}, IoU {val_iou:.4f} | "
            f"lr {learning_rate:.2e}"
        )

        if val_loss < best_val_loss - early_stopping_min_delta:
            best_val_loss = val_loss
            epochs_without_improvement = 0
            torch.save(unwrap_model(model).state_dict(), saved_model)
            print(f"Saved best model to {saved_model}")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= early_stopping_patience:
                print(
                    f"Early stopping at epoch {epoch + 1}: "
                    f"no validation-loss improvement for {early_stopping_patience} epochs."
                )
                break

    best_state_dict = torch.load(saved_model, map_location=device_override)
    unwrap_model(model).load_state_dict(best_state_dict)
    return unwrap_model(model), train_metrics, val_metrics

In [ ]:
def evaluate(model, dataloader, device_override=device):
    model.eval()
    criterion = BCEDiceLoss()
    total_tp = total_fp = total_fn = total_loss = 0.0

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device_override, non_blocking=pin_memory)
            masks = masks.to(device_override, non_blocking=pin_memory)
            outputs = model(images)
            total_loss += criterion(outputs, masks).item() * images.size(0)
            predictions = (outputs > 0).float()
            total_tp += (predictions * masks).sum().item()
            total_fp += (predictions * (1 - masks)).sum().item()
            total_fn += ((1 - predictions) * masks).sum().item()

    loss = total_loss / len(dataloader.dataset)
    precision = total_tp / (total_tp + total_fp + 1e-8)
    recall = total_tp / (total_tp + total_fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    iou = total_tp / (total_tp + total_fp + total_fn + 1e-8)

    print(
        f"Loss: {loss:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, "
        f"F1: {f1:.4f}, IoU: {iou:.4f}"
    )
    return loss, precision, recall, f1, iou


eval = evaluate

In [ ]:
import seaborn as sns

def plot_precision(train_precisions, val_precisions, model_suffix=""):
    fig = plt.figure(figsize=(10, 6))
    sns.lineplot(data=train_precisions, label="Train Precision", marker="o")
    sns.lineplot(data=val_precisions, label="Validation Precision", marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Precision")
    plt.title(f"Training and Validation Precision {model_suffix}")
    plt.ylim(0, 1)
    plt.legend()
    plt.tight_layout()
    save_plot(fig, f"precision_plot_{model_suffix}.png")

def plot_loss(train_losses, val_losses, model_suffix=""):
    fig = plt.figure(figsize=(10, 6))
    sns.lineplot(data=train_losses, label="Train Loss", marker="o")
    sns.lineplot(data=val_losses, label="Validation Loss", marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Training and Validation Loss {model_suffix}")
    plt.legend()
    plt.tight_layout()
    save_plot(fig, f"loss_plot_{model_suffix}.png")

def plot_recall(train_recalls, val_recalls, model_suffix=""):
    fig = plt.figure(figsize=(10, 6))
    sns.lineplot(data=train_recalls, label="Train Recall", marker="o")
    sns.lineplot(data=val_recalls, label="Validation Recall", marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Recall")
    plt.title(f"Training and Validation Recall {model_suffix}")
    plt.ylim(0, 1)
    plt.legend()
    plt.tight_layout()
    save_plot(fig, f"recall_plot_{model_suffix}.png")

def plot_f1(train_f1s, val_f1s, model_suffix=""):
    fig = plt.figure(figsize=(10, 6))
    sns.lineplot(data=train_f1s, label="Train F1-Score", marker="o")
    sns.lineplot(data=val_f1s, label="Validation F1-Score", marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("F1-Score")
    plt.title(f"Training and Validation F1-Score {model_suffix}")
    plt.ylim(0, 1)
    plt.legend()
    plt.tight_layout()
    save_plot(fig, f"f1_plot_{model_suffix}.png")

def plot_iou(train_ious, val_ious, model_suffix=""):
    fig = plt.figure(figsize=(10, 6))
    sns.lineplot(data=train_ious, label="Train IoU", marker="o")
    sns.lineplot(data=val_ious, label="Validation IoU", marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("IoU")
    plt.title(f"Training and Validation IoU {model_suffix}")
    plt.ylim(0, 1)
    plt.legend()
    plt.tight_layout()
    save_plot(fig, f"iou_plot_{model_suffix}.png")


In [ ]:
import subprocess
import sys


def train_with_ddp(model_name, learning_rate, num_epochs, seed=0):
    """Launch Lightning DDP, stream epoch metrics, and verify checkpoint output."""
    if not torch.cuda.is_available():
        raise RuntimeError("DDP training requires CUDA GPUs.")
    if gpu_count < 2:
        raise RuntimeError("At least two CUDA GPUs are required for this DDP launcher.")

    checkpoint_dir = CHECKPOINT_PATH.resolve()
    log_dir = LOG_PATH.resolve()
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    existing_checkpoints = {
        path.resolve()
        for pattern in ("*.pt", "*.ckpt")
        for path in checkpoint_dir.glob(pattern)
    }

    command = [
        sys.executable,
        str(base_dir / "train.py"),
        "--model",
        model_name,
        "--base-dir",
        str(base_dir),
        "--batch-size",
        str(batch_size_per_gpu),
        "--num-workers",
        str(num_workers),
        "--max-epochs",
        str(num_epochs),
        "--seeds",
        str(seed),
        "--lr",
        str(learning_rate),
        "--checkpoint-dir",
        str(checkpoint_dir),
        "--log-dir",
        str(log_dir),
        "--accelerator",
        "gpu",
        "--devices",
        str(ddp_world_size),
        "--log-every-n-steps",
        "1",
        "--no-download",
        "--no-validate",
        "--no-drive-cache",
    ]

    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    print("Starting Lightning DDP. Epoch metrics will be printed below.", flush=True)
    print(f"Best checkpoints will be saved in: {checkpoint_dir}", flush=True)
    process = subprocess.Popen(
        command,
        cwd=base_dir,
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for output_line in process.stdout:
        print(output_line, end="", flush=True)

    exit_code = process.wait()
    if exit_code:
        raise RuntimeError(f"DDP training failed with exit code {exit_code}.")

    saved_checkpoints = sorted(
        [
            path
            for pattern in ("*.pt", "*.ckpt")
            for path in checkpoint_dir.glob(pattern)
            if path.resolve() not in existing_checkpoints
        ],
        key=lambda path: path.stat().st_mtime,
    )
    if not saved_checkpoints:
        raise RuntimeError(
            f"Training completed but no checkpoint was written to {checkpoint_dir}."
        )
    for checkpoint in saved_checkpoints:
        print(f"Saved checkpoint: {checkpoint}")

In [ ]:
# Loop Multi-Seed: garante reprodutibilidade estatística (seeds 0, 1, 2).
# U-Net: 20 epochs | Attention U-Net: 60 epochs
seeds = [0, 1, 2]
unet_epochs = 20
attn_unet_epochs = 60

for seed in seeds:
    print(f"\n{'='*60}")
    print(f"  Treinando U-Net | seed={seed} | max_epochs={unet_epochs}")
    print(f"{'='*60}\n")
    train_with_ddp(
        model_name="unet",
        learning_rate=1e-4,
        num_epochs=unet_epochs,
        seed=seed,
    )

for seed in seeds:
    print(f"\n{'='*60}")
    print(f"  Treinando Attention U-Net | seed={seed} | max_epochs={attn_unet_epochs}")
    print(f"{'='*60}\n")
    train_with_ddp(
        model_name="attn_unet",
        learning_rate=5e-4,
        num_epochs=attn_unet_epochs,
        seed=seed,
    )

## Results

In [ ]:
import sys

if str(base_dir) not in sys.path:
    sys.path.insert(0, str(base_dir))

from src.models import get_model


def load_trained_model(model_name, weights_path):
    model = get_model(model_name, num_inputs=4, num_outputs=1)
    state_dict = torch.load(weights_path, map_location="cpu", weights_only=True)
    model.load_state_dict(state_dict)
    return model.to(device)


attention_weight_paths = sorted(CHECKPOINT_PATH.glob("attn_unet_seed_*_best.pt"))
if not attention_weight_paths:
    raise FileNotFoundError(
        f"No Attention U-Net weights found in {CHECKPOINT_PATH}. "
        "Expected files named attn_unet_seed_<seed>_best.pt."
    )

precisions_am4_val, recalls_am4_val, f1s_am4_val, ious_am4_val = [], [], [], []
precisions_am4_test, recalls_am4_test, f1s_am4_test, ious_am4_test = [], [], [], []

for weights_path in attention_weight_paths:
    model = load_trained_model("attn_unet", weights_path)
    print(f"Evaluating {weights_path.name}")

    loss, precision, recall, f1, iou = evaluate(model, val_dataloader_am4)
    precisions_am4_val.append(precision)
    recalls_am4_val.append(recall)
    f1s_am4_val.append(f1)
    ious_am4_val.append(iou)

    loss, precision, recall, f1, iou = evaluate(model, test_dataloader_am4)
    precisions_am4_test.append(precision)
    recalls_am4_test.append(recall)
    f1s_am4_test.append(f1)
    ious_am4_test.append(iou)

print("Validation Metrics for Attention U-Net on AM4:")
print(f"precision mean: {np.mean(precisions_am4_val):.4f}, precision std: {np.std(precisions_am4_val):.4f}")
print(f"recall mean: {np.mean(recalls_am4_val):.4f}, recall std: {np.std(recalls_am4_val):.4f}")
print(f"f1 mean: {np.mean(f1s_am4_val):.4f}, f1 std: {np.std(f1s_am4_val):.4f}")
print(f"iou mean: {np.mean(ious_am4_val):.4f}, iou std: {np.std(ious_am4_val):.4f}")

print("Test Metrics for Attention U-Net on AM4:")
print(f"precision mean: {np.mean(precisions_am4_test):.4f}, precision std: {np.std(precisions_am4_test):.4f}")
print(f"recall mean: {np.mean(recalls_am4_test):.4f}, recall std: {np.std(recalls_am4_test):.4f}")
print(f"f1 mean: {np.mean(f1s_am4_test):.4f}, f1 std: {np.std(f1s_am4_test):.4f}")
print(f"iou mean: {np.mean(ious_am4_test):.4f}, iou std: {np.std(ious_am4_test):.4f}")

In [ ]:
weights_path = attention_weight_paths[0]
model_attn = load_trained_model("attn_unet", weights_path)
model_attn.eval()
image, true_mask_tensor = train_dataset_am4[1]
image = torch.as_tensor(image, dtype=torch.float32).unsqueeze(0).to(device)
with torch.no_grad():
    pred = model_attn(image)
pred_mask = (pred > 0.0).float().squeeze().cpu().numpy()
fig = plt.figure(figsize=(12, 6))
plt.subplot(1, 3, 1)
img_np = image.squeeze().cpu().numpy()
if img_np.ndim == 3 and img_np.shape[0] == 4:
    rgb = np.transpose(img_np[:3], (1, 2, 0))
    plt.imshow(rgb)
else:
    plt.imshow(img_np.squeeze(), cmap="Greys_r")
plt.title("Input Image")
plt.subplot(1, 3, 2)
plt.imshow(pred_mask, cmap="gray")
plt.title("Detected Mask")
plt.subplot(1, 3, 3)
true_mask = true_mask_tensor.squeeze().cpu().numpy()
plt.imshow(true_mask, cmap="gray")
plt.title("True Mask")
save_plot(fig, "attention_unet_pred_true_mask.png")

In [ ]:
attention_log_dir = LOG_PATH / "attn_unet"
if attention_log_dir.exists():
    print(f"Attention U-Net training curves are stored in: {attention_log_dir.resolve()}")
else:
    print(f"No Attention U-Net TensorBoard logs found yet in: {attention_log_dir.resolve()}")

In [ ]:
unet_weight_paths = sorted(CHECKPOINT_PATH.glob("unet_seed_*_best.pt"))
if not unet_weight_paths:
    raise FileNotFoundError(
        f"No U-Net weights found in {CHECKPOINT_PATH}. "
        "Expected files named unet_seed_<seed>_best.pt."
    )

precisions_am4_val, recalls_am4_val, f1s_am4_val, ious_am4_val = [], [], [], []
precisions_am4_test, recalls_am4_test, f1s_am4_test, ious_am4_test = [], [], [], []

for weights_path in unet_weight_paths:
    model = load_trained_model("unet", weights_path)
    print(f"Evaluating {weights_path.name}")

    loss, precision, recall, f1, iou = evaluate(model, val_dataloader_am4)
    precisions_am4_val.append(precision)
    recalls_am4_val.append(recall)
    f1s_am4_val.append(f1)
    ious_am4_val.append(iou)

    loss, precision, recall, f1, iou = evaluate(model, test_dataloader_am4)
    precisions_am4_test.append(precision)
    recalls_am4_test.append(recall)
    f1s_am4_test.append(f1)
    ious_am4_test.append(iou)

print("Validation Metrics for U-Net on AM4:")
print(f"precision mean: {np.mean(precisions_am4_val):.4f}, precision std: {np.std(precisions_am4_val):.4f}")
print(f"recall mean: {np.mean(recalls_am4_val):.4f}, recall std: {np.std(recalls_am4_val):.4f}")
print(f"f1 mean: {np.mean(f1s_am4_val):.4f}, f1 std: {np.std(f1s_am4_val):.4f}")
print(f"iou mean: {np.mean(ious_am4_val):.4f}, iou std: {np.std(ious_am4_val):.4f}")

print("Test Metrics for U-Net on AM4:")
print(f"precision mean: {np.mean(precisions_am4_test):.4f}, precision std: {np.std(precisions_am4_test):.4f}")
print(f"recall mean: {np.mean(recalls_am4_test):.4f}, recall std: {np.std(recalls_am4_test):.4f}")
print(f"f1 mean: {np.mean(f1s_am4_test):.4f}, f1 std: {np.std(f1s_am4_test):.4f}")
print(f"iou mean: {np.mean(ious_am4_test):.4f}, iou std: {np.std(ious_am4_test):.4f}")

In [ ]:
weights_path = unet_weight_paths[0]
model = load_trained_model("unet", weights_path)
model.eval()
image, true_mask_tensor = train_dataset_am4[0]
image = torch.as_tensor(image, dtype=torch.float32).unsqueeze(0).to(device)
with torch.no_grad():
    pred = model(image)
pred_mask = (pred > 0.0).float().squeeze().cpu().numpy()
fig = plt.figure(figsize=(12, 6))
plt.subplot(1, 3, 1)
img_np = image.squeeze().cpu().numpy()
if img_np.ndim == 3 and img_np.shape[0] == 4:
    rgb = np.transpose(img_np[:3], (1, 2, 0))
    plt.imshow(rgb)
else:
    plt.imshow(img_np.squeeze(), cmap="Greys_r")
plt.title("Input Image")
plt.subplot(1, 3, 2)
plt.imshow(pred_mask, cmap="gray")
plt.title("Detected Mask")
plt.subplot(1, 3, 3)
true_mask = true_mask_tensor.squeeze().cpu().numpy()
plt.imshow(true_mask, cmap="gray")
plt.title("True Mask")
save_plot(fig, "unet_pred_true_mask.png")

In [ ]:
import pandas as pd

rows = []
for model_name, weights_paths in (
    ("Attention U-Net", attention_weight_paths),
    ("U-Net", unet_weight_paths),
):
    for weights_path in weights_paths:
        state_dict = torch.load(weights_path, map_location="cpu", weights_only=True)
        params = sum(value.numel() for value in state_dict.values() if torch.is_tensor(value))
        rows.append(
            {
                "model": model_name,
                "checkpoint": weights_path.name,
                "params": params,
                "size_mb": round(weights_path.stat().st_size / 1024**2, 2),
            }
        )

df = pd.DataFrame(rows).sort_values(["model", "checkpoint"])
display(df)

In [ ]:
import gc
from torch.utils.data import DataLoader

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

val_dataloader_am4 = DataLoader(
    val_dataset_am4,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
 )

images, masks = next(iter(val_dataloader_am4))

print("Device:", device)
print("Images shape:", images.shape)
print("Masks shape:", masks.shape)
print("Batch size:", val_dataloader_am4.batch_size)

weights_path = unet_weight_paths[0]
model = load_trained_model("unet", weights_path)
model.eval()

with torch.no_grad():
    images = images.to(device)
    output = model(images)

print("Output shape:", output.shape)

del output
del images
del masks
del model

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nTeste concluído sem OOM.")
if torch.cuda.is_available():
    print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"Reserved : {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

In [ ]:
import random

num_images = 50
weights_path = attention_weight_paths[0]
model_attn = load_trained_model("attn_unet", weights_path)
model_attn.eval()

dataset_len = len(val_dataset_am4)
num_images = min(num_images, dataset_len)
random_indices = random.sample(range(dataset_len), num_images)

print(f"Evaluating {num_images} random validation images with {weights_path.name}...")

with torch.no_grad():
    for idx in random_indices:
        image, true_mask_tensor = val_dataset_am4[idx]
        image_input = torch.as_tensor(image, dtype=torch.float32).unsqueeze(0).to(device)
        pred = model_attn(image_input)
        pred_mask = (pred > 0.0).float().squeeze().cpu().numpy()

        img_np = image_input.squeeze().cpu().numpy()
        if img_np.ndim == 3 and img_np.shape[0] in (3, 4):
            rgb = np.transpose(img_np[:3], (1, 2, 0))
        else:
            rgb = img_np.squeeze()
        true_mask = true_mask_tensor.squeeze().cpu().numpy()

        fig = plt.figure(figsize=(15, 5))
        plt.subplot(1, 3, 1)
        if rgb.ndim == 3:
            plt.imshow(np.clip(rgb, 0, 1))
        else:
            plt.imshow(rgb, cmap="Greys_r")
        plt.title(f"Input Image (Index: {idx})")
        plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.imshow(pred_mask, cmap="gray")
        plt.title("Detected Mask")
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.imshow(true_mask, cmap="gray")
        plt.title("True Mask")
        plt.axis("off")

        plt.tight_layout()
        plt.show()